# 03 — End-to-end paper figures & headline tables

This notebook is the paper-friendly entry point. It does **not** load any
model weights. It re-derives every figure and table that appears in
`paper/docs/paper/main.pdf` from the artifacts already committed under
`experiments/`.

Pipeline:

1. Run the aggregator to refresh the two CSVs:
   - `paper/notebooks/sweep_results.csv` — one row per (run, response).
   - `paper/notebooks/expert_selection.csv` — one row per (run, layer, expert).
2. Show the headline same-run paired delta table for question-only
   `Mixtral baseline` vs `Mixtral + SteerMoE`.
3. Show the diffuse-signal histogram and the steering-strength sweep.

If you only have access to the committed CSVs (no `experiments/` checkout),
skip cell 1 — the rest of the notebook still runs.

In [ ]:
import subprocess
from pathlib import Path

REPO = Path('.').resolve()
while REPO != REPO.parent and not (REPO / 'experiments').exists():
    REPO = REPO.parent

subprocess.run(['python3', 'paper/scripts/aggregate_results.py'], cwd=REPO, check=True)
subprocess.run(['python3', 'paper/scripts/make_figures.py'], cwd=REPO, check=True)
print('repo root:', REPO)

In [ ]:
import pandas as pd

sweep = pd.read_csv(REPO / 'paper' / 'notebooks' / 'sweep_results.csv')
experts = pd.read_csv(REPO / 'paper' / 'notebooks' / 'expert_selection.csv')
print('sweep rows:', len(sweep))
print('expert rows:', len(experts))
sweep.head(3)

## Headline same-run paired delta

One row per question-only run. `delta__concept_token_hit_rate` is the
fraction-of-cases shift (out of 25) and is the key diagnostic for Claim 2.

In [ ]:
delta = pd.read_csv(REPO / 'paper' / 'docs' / 'paper' / 'tables' / 'steermoe_vs_baseline_delta.csv')
show = delta[[
    'run_name', 'steering_coefficient', 'top_positive_experts', 'top_negative_experts',
    'mixtral_baseline__concept_token_hit_rate', 'mixtral_steermoe__concept_token_hit_rate',
    'delta__concept_token_hit_rate', 'delta__mean_word_count', 'delta__exceeds_20_words_rate',
]].copy()
show['run_name'] = show['run_name'].str.replace('mixtral_steermoe_fears_seed7_', '', regex=False)
show.sort_values(['steering_coefficient', 'top_positive_experts'])

## Diffuse-signal diagnostic

Top-20 share of $\sum |\Delta_{\ell,e}|$ per (run, concept). If this is
consistently below ~0.1 the routing-bias readout is too diffuse for the
sparse-plan intervention to dominate the unbiased router.

In [ ]:
concentration = pd.read_csv(REPO / 'paper' / 'docs' / 'paper' / 'tables' / 'expert_concentration.csv')
concentration.describe(percentiles=[0.1, 0.5, 0.9])

## Figures

These are the same PDFs the paper includes. They live under
`paper/docs/paper/figures/`. Re-run the second cell after editing
`paper/scripts/make_figures.py` to regenerate.

In [ ]:
from IPython.display import IFrame
IFrame(REPO / 'paper' / 'docs' / 'paper' / 'figures' / 'expert_concentration.pdf', width=600, height=400)

In [ ]:
IFrame(REPO / 'paper' / 'docs' / 'paper' / 'figures' / 'steering_strength_sweep.pdf', width=600, height=400)